# Score based diffusion models

## Теория


Если у нас есть SDE (Stochastic differencial equation):

$ \partial x = {\color{Aqua}f(x, t)}\partial t + {\color{red}g(t)}\partial \omega$

Где:

1) ${\color{Aqua}f(x, t)}\partial t$ представляет общее движение
2) ${\color{red}g(t)}\partial \omega$ представляет случайные процессы, по модулю ${\color{red}g(t)}$

Тогда для него существует обратное во времени уравнение (Reverse SDE):

$ \partial x = [{\color{Aqua}f(x, t)} - {\color{red}g^2(t)}{\color{DarkViolet}\nabla log(p_t(x))}]\partial t + {\color{red}g(t)}\partial \={\omega}$

Где ${\color{DarkViolet}\nabla log(p_t(x))}$ это то что называется "score function".


Мы можем использовать первое уравнение что бы зашумлить наши данные, и второе что бы восстановить. Т.е решая Reverse SDE для случайной точки, мы можем сэмплировать из распределения $\color{DarkViolet}p(x)$

Но какую взять функцию ${\color{Aqua}f(x, t)}$?

Мы можем использовать разные функции (я пока что не знаю какие еще кроме той что используется сейчас и сдесь), например метод зашумления из DDPM возможно переписать как SDE.
В DDPM определяем процесс по зашумлению данных как:

$x_{i + 1} = \sqrt{\alpha_i}x_i + \sqrt{1-\alpha_i}\epsilon_{i-1}$,
где $1 - \alpha$ это дисперсия

Почему так? Потому что определив шум таким образом, получается выразить любой шаг через i напрямую:

$x_i = \sqrt{\=\alpha_i}x_0 + \sqrt{1-\=\alpha_i}\epsilon_{i-1}$

Это можно увидеть если подставить $x_i$ в рекурентную формулу

Теперь введем новую переменную $\beta_i = 1 - \=\alpha_i$, тогда выражение становится

$x_{i + 1} = \sqrt{1 - \beta_i}x_{i} + \sqrt{\beta_i}\epsilon_{i-1}$
Теперь введем $\~\beta_i = \beta_iN$
Выражеине станвоится
$x_i = \sqrt{1 - \frac{\~\beta_i}{N}}x_i + \sqrt{\frac{\~\beta_i}{N}}\epsilon_{i-1}$

Теперь репеинтертируем индекс как функцию от времени, где $\varDelta t = \frac{1}{N}$

$x(t + \varDelta t) = \sqrt{1 - \beta_i(t + \varDelta t) \varDelta t}x(t) + \sqrt{\beta_i(t + \varDelta t) \varDelta t}\epsilon(t)$

Теперь, используем ряд тейлора для $\sqrt{x}$, упростив выражение (мы можем взять конечное число членов из ряда, поскольку полагаем что $\varDelta t \rightarrow 0$)

$x(t + \varDelta t) = \frac{1}{2}(2 - \beta(t + \varDelta t) \varDelta t)x(t) + \sqrt{\beta(t + \varDelta t) \varDelta t}\epsilon(t)$

$x(t + \varDelta t) = x(t) - x(t)\frac{1}{2}\beta(t + \varDelta t) \varDelta t + \sqrt{\beta(t + \varDelta t) \varDelta t}\epsilon(t)$

В итоге получаем

$\partial x = -x\frac{1}{2}\beta(t)\partial t + \sqrt{\beta(t) \partial t}\epsilon(t) = -x\frac{1}{2}\beta(t)\partial t + \sqrt{\beta(t)}\sqrt{\partial t}\epsilon(t)$

Почему то $\sqrt{\partial t}\epsilon(t) = \partial\omega$

$\partial x = -x\frac{1}{2}\beta(t)\partial t + \sqrt{\beta(t)}\partial\omega$

Тепер у нас есть SDE, которое можно реверснуть

$$
\color{Aqua}-x\frac{1}{2}\beta(t) = f(x, t) \\
\color{Red}\sqrt{\beta(t)} = g(t)
$$
Что бы решить уравнение назад во времени, нам нужно знать

${\color{DarkViolet}\nabla log(p_t(x))}$

Мы не знаем ее, но можем попытатся аппроксимировать ее, используя модель, минимизируя математическое ожидание 

$L = \mathbb{E}_{x_0\sim p(x)} [\lVert s_\theta - {\color{DarkViolet}\nabla log(p_t(x))}\rVert^2]$

Однако, мы не знаем его, по этому данное выражение не подходит.

К счатью, для достаточно большой выборки задача минимизации становится эквивалента

$L = \mathbb{E}_{x_0\sim p(x)} [\lVert s_\theta - {\color{DarkViolet}\nabla log(p_t(x | x_0))}\rVert^2]$

Это можно представить как если мы заставляем модель выучить направление к среднему всего распределения на больших уровнях шума, и "локальному" среднему на маленьких.
Т.е из $x_0$ в $x_t$ ведет бесконечное количество путей, т.е из разных примеров мы будем попадать в +- один и тот же $x_t$, и разнообразие семьи возможных $x_0$ будет зависить от уровня добавленного шума. Тогда, модель будет учится предсказывать среднее для конкретной семьи возможных $x_0$. Именно поэтому модели необходимо знать степень зашумленности, иначе она будет учится среднему всего датасета.